## Persistent Landing - Delta Lake (Non-Structured)

Even though Delta tables are typically used to store structured data, they can also be used to store metadata extracted from unstructured or semi-structured data.

**Importing Useful Libraries**

In [17]:
import os
import boto3
import duckdb
import hashlib
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl
from PIL import Image
from PIL.ExifTags import TAGS
import pandas as pd
import json
import io

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [18]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [19]:
def get_deep_keys(data,level=0):
    # If it's a list, dive into the first element
    if isinstance(data, list) and len(data) > 0:
        return get_deep_keys(data[0],level+1)

    # If it's finally a dictionary, return the keys
    if isinstance(data, dict):
        return list(data.keys()),level

    # If it's a primitive (like a string or number) or empty
    return [],level

def extract_timestamp_from_filename(filename):
    # Strip extension and split by underscore
    name_part = os.path.splitext(filename)[0]
    raw_ts = name_part.split('_')[-1]

    try:
        # Convert string epoch to a readable datetime object
        dt_object = datetime.fromtimestamp(int(raw_ts))
        return dt_object
    except (ValueError, IndexError):
        # Fallback if the filename doesn't follow the pattern
        return datetime.now()
        
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

**Semistructured Data**

In [20]:
def process_json(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            src_key = obj["Key"]

            # 1. Skip directories and empty files
            if src_key.endswith("/") or obj['Size'] == 0:
                continue

            print(f"Processing: {src_key}")

            # 2. Read and Parse
            response = s3.get_object(Bucket=bucket, Key=src_key)
            content = response['Body'].read().decode('utf-8')
            data = json.loads(content)

            # 3. Extract Deep Metadata
            keys_list, level = get_deep_keys(data)

            # 4. Correct Record Counting (Flattening for the count)
            # This ensures [[{}, {}]] returns 2, not 1
            temp_data = data
            for _ in range(level):
                if isinstance(temp_data, list) and len(temp_data) > 0:
                    temp_data = [item for sublist in temp_data for item in (sublist if isinstance(sublist, list) else [sublist])]
            record_count = len(temp_data)

            # 5. Build the Metadata Blob (The "Table inside a Table")
            # This blob changes structure based on file type
            metadata_blob = {
                "nesting_level": level,
                "schema_keys": keys_list,
                "file_size_bytes": obj['Size']
            }

            # 6. Prepare Final Catalog Row
            filename = os.path.basename(src_key)
            metadata_row = pd.DataFrame([{
                "file_id": filename,
                "source_type": src_key.split('/')[2],
                "file_type": "JSON",
                "event_time": extract_timestamp_from_filename(filename),
                "record_count": record_count,
                "metadata_blob": json.dumps(metadata_blob),  # The flexible packet
                "processed_at": pd.Timestamp.now()
            }])

            # 7. Append to Master Catalog
            write_deltalake(
                "s3://landing-zone/persistent-landing/structured/file_catalog/",
                metadata_row,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )

In [21]:
process_json("landing-zone", "persistent-landing/semistructured/")


Processing: persistent-landing/semistructured/airquality-barcelona.json
Processing: persistent-landing/semistructured/weather-barcelona.json


In [22]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

In [23]:
# 1. Show all columns (don't hide the middle ones)
pd.set_option('display.max_columns', None)

# 2. Show the full content of each cell (don't truncate long JSON/strings)
pd.set_option('display.max_colwidth', None)

# 3. Show all rows (optional: only use if the table is small, e.g., < 100 rows)
# pd.set_option('display.max_rows', None)

# Execute and display
query = "SELECT * FROM delta_scan('s3://landing-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,source_type,file_type,event_time,record_count,metadata_blob,processed_at
0,weather-barcelona.json,weather-barcelona.json,JSON,2026-04-03 15:19:04.589363,33,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 4620}",2026-04-03 15:19:04.589377
1,airquality-barcelona.json,airquality-barcelona.json,JSON,2026-04-03 15:19:04.381856,102,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 187238}",2026-04-03 15:19:04.381880
2,weather-barcelona.json,weather-barcelona.json,JSON,2026-04-03 15:09:34.035668,33,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 4620}",2026-04-03 15:09:34.035694
3,airquality-barcelona.json,airquality-barcelona.json,JSON,2026-04-03 15:09:33.877903,102,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 187238}",2026-04-03 15:09:33.877955


**Unstructured Data**

In [24]:
def process_image(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        page_records = []

        for obj in page.get("Contents", []):
            try:
                src_key = obj["Key"]
                # 1. Skip directories and empty files
                if src_key.endswith("/") or obj['Size'] == 0:
                    continue

                # 2. Read and Parse
                response = s3.get_object(Bucket=bucket, Key=src_key)
                content = response['Body'].read()

                img = Image.open(io.BytesIO(content))
                metadata = response.get('Metadata', {})
                width, height = img.size

                metadata_blob = {
                                    "label": metadata.get('label'),
                                    "url": metadata.get('url'),
                                    "file_size_bytes": obj['Size'],
                                    "content_type": response.get('ContentType'),
                                    "width": width,
                                    "height": height,
                                    "aspect_ratio": round(width / height, 2) if height > 0 else 0,
                                    "image_mode": img.mode,
                                    "is_corrupted": False,
                                    "md5": hashlib.md5(content).hexdigest() # for duplicate detection
                                }

            except Exception as e:
                print(f"Error parsing image {src_key}: {e}")
                metadata_blob.update({
                    "is_corrupted": True,
                    "error_msg": str(e),
                    "width": 0, "height": 0, "aspect_ratio": 0, "image_mode": "unknown"
                })



            # 6. Prepare Final Catalog Row
            filename = os.path.basename(src_key)
            metadata_row = {
                "file_id": filename,
                "source_type": metadata.get('source'),
                "file_type": "Image",
                "event_time": extract_timestamp_from_filename(filename),
                "record_count": 1, # For image data it alaways 1
                "metadata_blob": json.dumps(metadata_blob),  # The flexible packet
                "processed_at": pd.Timestamp.now()
            }

            page_records.append(metadata_row)

        # 7. Append to Master Catalog
        if page_records:
            metadata_df = pd.DataFrame(page_records)
            write_deltalake(
                "s3://landing-zone/persistent-landing/structured/file_catalog/",
                metadata_df,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )
        print(f"Batch uploaded: {len(page_records)} images.")

In [25]:
process_image("landing-zone","persistent-landing/unstructured/image")

Batch uploaded: 999 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 1000 images.
Batch uploaded: 724 images.


In [26]:
query = "SELECT * FROM delta_scan('s3://landing-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,source_type,file_type,event_time,record_count,metadata_blob,processed_at
0,image_1775215950418.jpg,kaggle,Image,2026-04-03 15:20:16.789876,1,"{""label"": ""hail"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 57665, ""content_type"": ""image/jpg"", ""width"": 450, ""height"": 316, ""aspect_ratio"": 1.42, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""a52eb334f13bf60257346da53adb11a3""}",2026-04-03 15:20:16.789914
1,image_1775215950463.jpg,kaggle,Image,2026-04-03 15:20:16.793928,1,"{""label"": ""hail"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 37985, ""content_type"": ""image/jpg"", ""width"": 400, ""height"": 300, ""aspect_ratio"": 1.33, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""9655a6cceb1c72217b7b7bff8c839642""}",2026-04-03 15:20:16.793941
2,image_1775215950504.jpg,kaggle,Image,2026-04-03 15:20:16.798810,1,"{""label"": ""hail"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 50331, ""content_type"": ""image/jpg"", ""width"": 650, ""height"": 417, ""aspect_ratio"": 1.56, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""d61a992f722dd17d335ed0aa6ea734f8""}",2026-04-03 15:20:16.798820
3,image_1775215950539.jpg,kaggle,Image,2026-04-03 15:20:16.803245,1,"{""label"": ""hail"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 40410, ""content_type"": ""image/jpg"", ""width"": 400, ""height"": 266, ""aspect_ratio"": 1.5, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""fcf971675ebfadeb510f25b151f61d24""}",2026-04-03 15:20:16.803257
4,image_1775215950572.jpg,kaggle,Image,2026-04-03 15:20:16.807331,1,"{""label"": ""hail"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 19694, ""content_type"": ""image/jpg"", ""width"": 260, ""height"": 360, ""aspect_ratio"": 0.72, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""aa6b47c8be59f24e47222f86364f8e97""}",2026-04-03 15:20:16.807341
...,...,...,...,...,...,...,...
13722,image_1775214070331.jpg,kaggle,Image,2026-04-03 15:19:12.847757,1,"{""label"": ""sandstorm"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 19076, ""content_type"": ""image/jpg"", ""width"": 400, ""height"": 277, ""aspect_ratio"": 1.44, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""1c7793e2ff14c13b563cb2dbcecdd89b""}",2026-04-03 15:19:12.847768
13723,weather-barcelona.json,weather-barcelona.json,JSON,2026-04-03 15:19:04.589363,33,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 4620}",2026-04-03 15:19:04.589377
13724,airquality-barcelona.json,airquality-barcelona.json,JSON,2026-04-03 15:19:04.381856,102,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 187238}",2026-04-03 15:19:04.381880
13725,weather-barcelona.json,weather-barcelona.json,JSON,2026-04-03 15:09:34.035668,33,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 4620}",2026-04-03 15:09:34.035694
